# 第五次作业：sklearn 感知机源码解析（`Perceptron`）

> 主题：阅读 scikit-learn 的实现。

> **主线**：只讲 `linear_model.Perceptron`；PassiveAggressive 与 hinge-SVM 仅作对照，不展开。

环境：`scikit-learn 1.3.2` / Python 3.8。文中行号均以 **1.3.2** 版本的库为准。

一句话贯穿全篇：**`Perceptron` 不是独立算法，而是 `BaseSGDClassifier` 的一组锁死默认值 + 损失表里 `Hinge(θ=0)` 这一行。**

## 1. 感知机

教科书版：对每个样本 $(x_i, y_i)$，$y_i \in \{-1,+1\}$，若 $y_i(w^\top x_i + b) \le 0$ 就更新

$$w \leftarrow w + \eta\, y_i x_i,\qquad b \leftarrow b + \eta\, y_i$$

课堂印象里的感知机：误分类驱动、无正则、不加权、没有概率输出、线性可分时才保证收敛。

而 sklearn 的 `Perceptron` 只有 221 行，其中 docstring 占 158 行。它真正做的事情只有两件：**裁剪参数** + **委托基类**。本作业按 P1–P10 逐条拆开。

## 2. 定位源码：221 行里的构成

`Perceptron` 的全部实现都在 `sklearn/linear_model/_perceptron.py`，它不新增任何训练逻辑：改参数约束表、写死几个默认值，然后调用 `BaseSGDClassifier`。

`linear_model/` 目录下的 `.py` 都可 `inspect.getsource`；`.so` 只有二进制；`.pxd` 只是给 `_sag_fast` 用的声明，**不含 Hinge**。

In [1]:
import ast
import inspect
import os
import sys

import sklearn
from sklearn.linear_model import Perceptron

sk_dir = os.path.dirname(sklearn.__file__)
lm_dir = os.path.join(sk_dir, "linear_model")
path = inspect.getsourcefile(Perceptron)

print("python       :", sys.version.split()[0])
print("sklearn      :", sklearn.__version__)
print("perceptron   :", path)
print()

tree = ast.parse(open(path, encoding="utf-8").read())
cls = next(n for n in tree.body if isinstance(n, ast.ClassDef) and n.name == "Perceptron")
doc = cls.body[0]
total = cls.end_lineno - cls.lineno + 1
doc_lines = doc.end_lineno - doc.lineno + 1

print("== Perceptron 类的行数构成 ==")
print("  整个类   : %d 行（L%d–L%d）" % (total, cls.lineno, cls.end_lineno))
print("  docstring: %d 行（L%d–L%d）" % (doc_lines, doc.lineno, doc.end_lineno))
print("  其余     : %d 行（含导入与空行）" % (total - doc_lines))
print()

print("== linear_model 目录 ==")
for suffix, label in ((".py", "纯 Python"), (".so", "编译产物"), (".pxd", "Cython 声明")):
    files = sorted(f for f in os.listdir(lm_dir) if f.endswith(suffix))
    print("%-10s (%d): %s" % (label, len(files), ", ".join(files) if files else "-"))

python       : 3.8.20
sklearn      : 1.3.2
perceptron   : /root/MSA/ENTER/envs/vmunet/lib/python3.8/site-packages/sklearn/linear_model/_perceptron.py

== Perceptron 类的行数构成 ==
  整个类   : 213 行（L9–L221）
  docstring: 158 行（L10–L167）
  其余     : 55 行（含导入与空行）

== linear_model 目录 ==
纯 Python   (17): __init__.py, _base.py, _bayes.py, _coordinate_descent.py, _huber.py, _least_angle.py, _linear_loss.py, _logistic.py, _omp.py, _passive_aggressive.py, _perceptron.py, _quantile.py, _ransac.py, _ridge.py, _sag.py, _stochastic_gradient.py, _theil_sen.py
编译产物       (3): _cd_fast.cpython-38-x86_64-linux-gnu.so, _sag_fast.cpython-38-x86_64-linux-gnu.so, _sgd_fast.cpython-38-x86_64-linux-gnu.so
Cython 声明  (1): _sgd_fast.pxd


## 3. P1｜减法 API：删掉 `loss` 与 `average`，再写死四个默认值

`Perceptron` 不是「另写一遍感知机」，而是把 `BaseSGDClassifier` 的能力**裁剪**成一个固定档位：

- `_parameter_constraints.pop("loss")`：用户不能改损失；
- `_parameter_constraints.pop("average")`：用户不能开权重平均（ASGD）；
- `__init__` 硬编码 `loss="perceptron"`、`learning_rate="constant"`、`eta0=1.0`、`power_t=0.5`。

这样 `Perceptron()` 就等价于 `SGDClassifier(loss="perceptron", eta0=1, learning_rate="constant", penalty=None)`。

还有一个容易踩的坑：**参数校验时机不一致**。`loss` / `average` 因为根本不在签名里，是**构造期** `TypeError`；而 `eta0=0` / `alpha=-1` 这类越界值是**构造通过、`fit` 期才报错**（sklearn 1.3 的参数校验挂在 `fit` 的装饰器上）。先看源码，再实测。

In [2]:
import inspect

from sklearn.linear_model import Perceptron

MARKS = {
    'self._parameter_constraints.pop("loss")': "  # <== 从公开 API 里删掉 loss",
    'self._parameter_constraints.pop("average")': "  # <== 从公开 API 里删掉 average",
    'loss="perceptron",': "  # <== 锁死损失",
    'learning_rate="constant",': "  # <== 锁死学习率表",
    'eta0=1.0,': "  # <== 默认步长 1.0（签名里可改）",
    'power_t=0.5,': "  # <== constant 下不生效，但仍写死",
}

src = inspect.getsource(Perceptron).splitlines()
start = next(i for i, l in enumerate(src) if l.strip().startswith("_parameter_constraints"))
end = next(i for i, l in enumerate(src) if l.strip().startswith("def __init__"))
print("== 参数约束表（类体片段）==")
for line in src[start:end]:
    print(line + MARKS.get(line.strip(), ""))

print()
print("== __init__ ==")
for line in inspect.getsource(Perceptron.__init__).splitlines():
    print(line + MARKS.get(line.strip(), ""))

c = Perceptron._parameter_constraints
print()
print("== 结果 ==")
print("  'loss' in constraints   :", "loss" in c)
print("  'average' in constraints:", "average" in c)
print("  penalty 允许值          :", c["penalty"])
print("  eta0 允许范围           :", c["eta0"])

== 参数约束表（类体片段）==
    _parameter_constraints: dict = {**BaseSGDClassifier._parameter_constraints}
    _parameter_constraints.pop("loss")
    _parameter_constraints.pop("average")
    _parameter_constraints.update(
        {
            "penalty": [StrOptions({"l2", "l1", "elasticnet"}), None],
            "alpha": [Interval(Real, 0, None, closed="left")],
            "l1_ratio": [Interval(Real, 0, 1, closed="both")],
            "eta0": [Interval(Real, 0, None, closed="left")],
        }
    )


== __init__ ==
    def __init__(
        self,
        *,
        penalty=None,
        alpha=0.0001,
        l1_ratio=0.15,
        fit_intercept=True,
        max_iter=1000,
        tol=1e-3,
        shuffle=True,
        verbose=0,
        eta0=1.0,  # <== 默认步长 1.0（签名里可改）
        n_jobs=None,
        random_state=0,
        early_stopping=False,
        validation_fraction=0.1,
        n_iter_no_change=5,
        class_weight=None,
        warm_start=False,
    ):
        super().__init__(
  

In [3]:
import warnings

import numpy as np
from sklearn.datasets import make_blobs
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Perceptron

X, y = make_blobs(n_samples=80, centers=2, random_state=0)

CASES = [
    ("average=True", dict(average=True)),
    ("loss='hinge'", dict(loss="hinge")),
    ("eta0=0.0", dict(eta0=0.0)),
    ("alpha=-1.0", dict(alpha=-1.0)),
    ("penalty='l1'", dict(penalty="l1")),
]

print("== 五种参数写法的报错时机 ==")
for label, kw in CASES:
    try:
        est = Perceptron(**kw)
    except Exception as e:
        print("%-14s -> 构造期 %-20s %s" % (label, type(e).__name__, str(e).splitlines()[0][:70]))
        continue
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            est.fit(X, y)
        print("%-14s -> 构造通过，fit 成功（coef[:2]=%s）" % (label, np.round(est.coef_[0][:2], 3)))
    except Exception as e:
        print("%-14s -> 构造通过，fit 期 %-18s %s" % (label, type(e).__name__, str(e).splitlines()[0][:70]))

print()
base = Perceptron(random_state=0).fit(X, y)
l1 = Perceptron(penalty="l1", random_state=0, max_iter=200).fit(X, y)
print("默认 penalty=None 的 coef[:2]:", np.round(base.coef_[0][:2], 3))
print("penalty='l1'      的 coef[:2]:", np.round(l1.coef_[0][:2], 3), "-> 已带正则，不是教科书更新")

== 五种参数写法的报错时机 ==
average=True   -> 构造期 TypeError            __init__() got an unexpected keyword argument 'average'
loss='hinge'   -> 构造期 TypeError            __init__() got an unexpected keyword argument 'loss'
eta0=0.0       -> 构造通过，fit 期 ValueError         eta0 must be > 0
alpha=-1.0     -> 构造通过，fit 期 InvalidParameterError The 'alpha' parameter of Perceptron must be a float in the range [0.0,
penalty='l1'   -> 构造通过，fit 成功（coef[:2]=[ 3.851 -8.56 ]）

默认 penalty=None 的 coef[:2]: [ 3.955 -8.664]
penalty='l1'      的 coef[:2]: [ 3.851 -8.56 ] -> 已带正则，不是教科书更新


## 4. P2｜「感知机损失」就是 `Hinge(θ=0)`

`loss_functions` 表里两行紧挨着：

```python
"hinge":      (Hinge, 1.0),
"perceptron": (Hinge, 0.0),
```

同一个 Cython 类，只差一个 `double` 阈值：`Hinge(0)` 是「零分界」，`Hinge(1)` 是「间隔 1 的线性 SVM」。

更硬核的是：`loss` / `dloss` 是 `cdef` 方法，Python 侧没有这两个名字——编译模块只额外暴露了专供测试的 `py_loss` / `py_dloss` 包装。正好可以**直接读出**判据，不必靠行为反推。

In [4]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron
from sklearn.linear_model._stochastic_gradient import BaseSGDClassifier

print("== loss_functions 表 ==")
for name, spec in BaseSGDClassifier.loss_functions.items():
    cls, args = spec[0], spec[1:]
    print("  %-28s -> %s%s" % (name, cls.__name__, args if args else ""))

X, y = make_blobs(n_samples=40, centers=2, random_state=0)
lf = Perceptron().fit(X, y).loss_function_
print()
print("Perceptron().loss_function_ :", type(lf).__module__ + "." + type(lf).__name__)
print("Python 侧有 .loss 吗？      :", hasattr(lf, "loss"))
print("Python 侧公开属性           :", [a for a in dir(lf) if not a.startswith("__")])
print("py_loss 的 self 说明        :", lf.py_loss.__doc__.splitlines()[0])

== loss_functions 表 ==
  hinge                        -> Hinge(1.0,)
  squared_hinge                -> SquaredHinge(1.0,)
  perceptron                   -> Hinge(0.0,)
  log_loss                     -> Log
  modified_huber               -> ModifiedHuber
  squared_error                -> SquaredLoss
  huber                        -> Huber(0.1,)
  epsilon_insensitive          -> EpsilonInsensitive(0.1,)
  squared_epsilon_insensitive  -> SquaredEpsilonInsensitive(0.1,)

Perceptron().loss_function_ : sklearn.linear_model._sgd_fast.Hinge
Python 侧有 .loss 吗？      : False
Python 侧公开属性           : ['py_dloss', 'py_loss']
py_loss 的 self 说明        : Python version of `loss` for testing.


In [5]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron, SGDClassifier

X, y = make_blobs(n_samples=40, centers=2, random_state=0)
lf_p = Perceptron().fit(X, y).loss_function_                  # Hinge(0.0)
lf_h = SGDClassifier(loss="hinge").fit(X, y).loss_function_    # Hinge(1.0)

print("== 直接读编译模块里的判据（py_loss / py_dloss）==")
print("  p 是 w^T x + intercept，z = p * y")
print()
print("%6s %3s | %10s %11s | %10s %11s" % ("p", "y", "P.py_loss", "P.py_dloss", "H.py_loss", "H.py_dloss"))
for p, yy in [(1.5, 1.0), (0.5, 1.0), (0.0, 1.0), (-0.5, 1.0), (-1.5, 1.0), (0.5, -1.0)]:
    print("%6.2f %3.0f | %10.3f %11.3f | %10.3f %11.3f"
          % (p, yy, lf_p.py_loss(p, yy), lf_p.py_dloss(p, yy),
             lf_h.py_loss(p, yy), lf_h.py_dloss(p, yy)))

print()
print("Perceptron 的 dloss 在 z <= 0 时非零 -> 分界恒为 0（z=0 也算）")
print("hinge 的 dloss 在 z < 1 时非零      -> 分界是间隔 1")

== 直接读编译模块里的判据（py_loss / py_dloss）==
  p 是 w^T x + intercept，z = p * y

     p   y |  P.py_loss  P.py_dloss |  H.py_loss  H.py_dloss
  1.50   1 |      0.000       0.000 |      0.000       0.000
  0.50   1 |      0.000       0.000 |      0.500      -1.000
  0.00   1 |      0.000      -1.000 |      1.000      -1.000
 -0.50   1 |      0.500      -1.000 |      1.500      -1.000
 -1.50   1 |      1.500      -1.000 |      2.500      -1.000
  0.50  -1 |      0.500       1.000 |      1.500       1.000

Perceptron 的 dloss 在 z <= 0 时非零 -> 分界恒为 0（z=0 也算）
hinge 的 dloss 在 z < 1 时非零      -> 分界是间隔 1


In [6]:
import numpy as np
from sklearn.linear_model import Perceptron, SGDClassifier


def probe(margin, estimator):
    X = np.array([[1.0], [5.0]])
    y = np.array([1, 0])
    sw = np.array([1.0, 0.0])
    est = estimator.fit(
        X, y,
        sample_weight=sw,
        coef_init=np.array([0.0]),
        intercept_init=np.array([float(margin)]),
    )
    return est.coef_.ravel().copy(), est.intercept_.copy()


print("== 同一初始决策值下，是否更新？==")
print("%8s | %-30s | %-30s" % ("margin", "Perceptron (theta=0)", "hinge (theta=1)"))
for m in (0.5, 2.0, -0.5):
    cp, bp = probe(m, Perceptron(shuffle=False, max_iter=1, tol=None))
    ch, bh = probe(m, SGDClassifier(loss="hinge", penalty=None, learning_rate="constant",
                                    eta0=1.0, shuffle=False, max_iter=1, tol=None))
    print("%8.2f | coef=%-8s b=%5.2f %-4s | coef=%-8s b=%5.2f %-4s"
          % (m, np.round(cp, 2), bp[0], "更新" if not np.allclose(cp, 0) else "不动",
             np.round(ch, 2), bh[0], "更新" if not np.allclose(ch, 0) else "不动"))

print()
print("== 边界：决策值恰好为 0 ==")
X0 = np.array([[0.0], [0.0]])
y0 = np.array([1, 0])
sw0 = np.array([1.0, 0.0])
p0 = Perceptron(shuffle=False, max_iter=1, tol=None).fit(
    X0, y0, sample_weight=sw0,
    coef_init=np.array([0.0]), intercept_init=np.array([0.0]))
print("  Perceptron intercept_ =", p0.intercept_, "-> 0 分界也会更新（<= 0）")

== 同一初始决策值下，是否更新？==
  margin | Perceptron (theta=0)           | hinge (theta=1)               
    0.50 | coef=[0.]     b= 0.50 不动   | coef=[1.]     b= 1.50 更新  
    2.00 | coef=[0.]     b= 2.00 不动   | coef=[0.]     b= 2.00 不动  
   -0.50 | coef=[1.]     b= 0.50 更新   | coef=[1.]     b= 0.50 更新  

== 边界：决策值恰好为 0 ==
  Perceptron intercept_ = [1.] -> 0 分界也会更新（<= 0）


## 5. P3｜教科书循环可以逐元素复现（条件很窄）

只有把下面五个开关同时凑齐，算法才退化成课堂上那条循环：

- `shuffle=False`（不洗牌）
- `tol=None`（不按 loss 早停，跑满 `max_iter`）
- 稠密输入（避开 P8 的截距衰减）
- `penalty=None`（无正则）
- `early_stopping=False`

此时更新规则就是 `if y*(x@w+b) <= 0: w += y*x; b += y`。注意是 **`<=`**：决策值恰好为 0 时也会更新，这一点用单样本实验单独确认。

In [7]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron


def manual_perceptron(X, y, max_epochs=1000):
    w = np.zeros(X.shape[1])
    b = 0.0
    updates = 0
    for epoch in range(1, max_epochs + 1):
        mistakes = 0
        for xi, yi in zip(X, y):
            if yi * (xi @ w + b) <= 0:
                w += yi * xi
                b += yi
                mistakes += 1
                updates += 1
        if mistakes == 0:
            return w, b, epoch, updates
    return w, b, max_epochs, updates


X, y = make_blobs(n_samples=200, centers=2, random_state=7, cluster_std=2.0)
y = np.where(y == 0, -1, 1)

w_m, b_m, epochs_m, updates_m = manual_perceptron(X, y)
sk = Perceptron(tol=None, shuffle=False, random_state=0, max_iter=1000).fit(X, y)

print("== 非可分数据：手工循环 vs sklearn ==")
print("  手工   : epochs=%d, updates=%d, coef[:2]=%s, b=%.4f"
      % (epochs_m, updates_m, np.round(w_m[:2], 4), b_m))
print("  sklearn: n_iter_=%d, t_=%d, coef[:2]=%s, b=%.4f"
      % (sk.n_iter_, int(sk.t_), np.round(sk.coef_[0][:2], 4), sk.intercept_[0]))
print("  逐元素一致 :", np.allclose(w_m, sk.coef_[0]) and np.allclose(b_m, sk.intercept_[0]))
print("  更新次数 vs t_ :", updates_m, "vs", int(sk.t_))
print()

print("== 边界：决策值 = 0 时是否更新 ==")
X0 = np.array([[0.0], [0.0]])
y0 = np.array([1, 0])
sw0 = np.array([1.0, 0.0])
p0 = Perceptron(shuffle=False, max_iter=1, tol=None).fit(
    X0, y0, sample_weight=sw0,
    coef_init=np.array([0.0]), intercept_init=np.array([0.0]))
print("  perceptron intercept_ =", p0.intercept_, "（从 0 变成 1 -> 0 分界也更新）")

for op in ("<", "<="):
    w = np.zeros(2)
    b = 0.0
    for _ in range(1000):
        mistakes = 0
        for xi, yi in zip(X, y):
            cond = yi * (xi @ w + b) < 0 if op == "<" else yi * (xi @ w + b) <= 0
            if cond:
                w += yi * xi
                b += yi
                mistakes += 1
        if mistakes == 0:
            break
    print("  手工用 '%s' 与 sklearn 一致: %s" % (op, bool(np.allclose(w, sk.coef_[0]) and np.allclose(b, sk.intercept_[0]))))

print()
Xs, ys = make_blobs(n_samples=60, centers=2, random_state=1, cluster_std=0.4)
print("== 可分数据：tol 决定跑几轮 ==")
print("  默认 tol=1e-3 : n_iter_ =", Perceptron(shuffle=False, random_state=0).fit(Xs, ys).n_iter_)
print("  tol=None      : n_iter_ =", Perceptron(shuffle=False, random_state=0, tol=None).fit(Xs, ys).n_iter_)

== 非可分数据：手工循环 vs sklearn ==
  手工   : epochs=1000, updates=14526, coef[:2]=[37.3739 -4.354 ], b=226.0000
  sklearn: n_iter_=1000, t_=200001, coef[:2]=[37.3739 -4.354 ], b=226.0000
  逐元素一致 : True
  更新次数 vs t_ : 14526 vs 200001

== 边界：决策值 = 0 时是否更新 ==
  perceptron intercept_ = [1.] （从 0 变成 1 -> 0 分界也更新）
  手工用 '<' 与 sklearn 一致: False


  手工用 '<=' 与 sklearn 一致: True

== 可分数据：tol 决定跑几轮 ==
  默认 tol=1e-3 : n_iter_ = 7
  tol=None      : n_iter_ = 1000


## 6. P4｜`t_` 的 docstring 在撒谎：它数的是样本，不是权重更新次数

`Perceptron` 的属性文档写：

> *Number of weight updates performed during training. Same as `(n_iter_ * n_samples + 1)`.*

后半句自己就暴露了问题：`n_iter_ * n_samples + 1` 是**处理过的样本数 + 1**，不是真正发生权重更新的次数。P3 的实验里，200 样本跑了 1000 轮（`t_=200001`），但只更新了 14526 次。

`fit` 每次都把 `t_` 重置为 1.0；`partial_fit` 不重置，跨调用累加。

In [8]:
import warnings

import numpy as np
from sklearn.datasets import load_digits, make_blobs
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Perceptron

X, y = make_blobs(n_samples=60, centers=2, random_state=0)
p = Perceptron(random_state=0).fit(X, y)
print("== 二分类 ==")
print("  n_iter_=%d, n_samples=%d, t_=%d, 公式值=%d"
      % (p.n_iter_, len(y), int(p.t_), p.n_iter_ * len(y) + 1))

early = Perceptron(early_stopping=True, random_state=0, max_iter=1000).fit(X, y)
print("  early_stopping: n_iter_=%d, t_=%d, 公式值=%d"
      % (early.n_iter_, int(early.t_), early.n_iter_ * len(y) + 1))

Xd, yd = load_digits(return_X_y=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    m = Perceptron(random_state=0, max_iter=50).fit(Xd, yd)
print("== 多分类（digits）==")
print("  n_iter_=%d, n_samples=%d, t_=%d, 公式值=%d"
      % (m.n_iter_, len(yd), int(m.t_), m.n_iter_ * len(yd) + 1))

print()
print("== partial_fit：t_ 累积，n_iter_ 恒为 1 ==")
pf = Perceptron(random_state=0)
pf.partial_fit(X, y, classes=np.array([0, 1]))
print("  第 1 次：n_iter_=%d, t_=%d" % (pf.n_iter_, int(pf.t_)))
pf.partial_fit(X, y)
print("  第 2 次：n_iter_=%d, t_=%d" % (pf.n_iter_, int(pf.t_)))

print()
print("== warm_start：coef 复用，但 t_ 仍被重置 ==")
ws = Perceptron(warm_start=True, random_state=0, max_iter=2, tol=None).fit(X, y)
t1 = int(ws.t_)
ws.fit(X, y)
print("  第 1 次 fit t_=%d, 第 2 次 fit t_=%d" % (t1, int(ws.t_)))

== 二分类 ==
  n_iter_=15, n_samples=60, t_=901, 公式值=901
  early_stopping: n_iter_=7, t_=421, 公式值=421


== 多分类（digits）==
  n_iter_=22, n_samples=1797, t_=39535, 公式值=39535

== partial_fit：t_ 累积，n_iter_ 恒为 1 ==
  第 1 次：n_iter_=1, t_=61
  第 2 次：n_iter_=1, t_=121

== warm_start：coef 复用，但 t_ 仍被重置 ==
  第 1 次 fit t_=121, 第 2 次 fit t_=121


## 7. P5｜多分类 = OvA + joblib：`n_jobs` 只在多分类生效

`Perceptron` 没有 softmax、也没有多类联合更新。`_partial_fit` 在 `n_classes > 2` 时走 `_fit_multiclass`：

- 每个类各训一个「本类 vs 其余」的二分类器；
- 先在外层用**原始多分类标签**算好验证掩码，所有 OvA job 共享；
- `Parallel(n_jobs=..., require="sharedmem")` 并行，每个 job 的随机种子在并行前统一抽取；
- 最后 `n_iter_` 取各二分类的**最大值**，不是总和。

因此 `n_jobs` 在二分类路径上完全没用——`_fit_binary` 的源码里根本不读它。

In [9]:
import inspect
import warnings

import numpy as np
from sklearn.datasets import load_digits
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Perceptron
from sklearn.linear_model._stochastic_gradient import BaseSGDClassifier

Xd, yd = load_digits(return_X_y=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    m = Perceptron(random_state=0, max_iter=50).fit(Xd, yd)

print("== digits 多分类 ==")
print("  coef_.shape      :", m.coef_.shape)
print("  intercept_.shape :", m.intercept_.shape)
print("  n_iter_          :", m.n_iter_, "（各二分类的最大值）")
print("  t_               :", int(m.t_), "=", m.n_iter_ * len(yd), "+ 1")
print()

print("== n_jobs 的作用范围 ==")
print("  '_fit_binary' 源码里出现 n_jobs ?", "n_jobs" in inspect.getsource(BaseSGDClassifier._fit_binary))
m1 = Perceptron(random_state=0, n_jobs=1, max_iter=10, tol=None).fit(Xd, yd)
m2 = Perceptron(random_state=0, n_jobs=-1, max_iter=10, tol=None).fit(Xd, yd)
print("  n_jobs=1 与 -1 的 coef_ 完全一致:", np.array_equal(m1.coef_, m2.coef_))
print("  两者 n_iter_:", m1.n_iter_, m2.n_iter_)

== digits 多分类 ==
  coef_.shape      : (10, 64)
  intercept_.shape : (10,)
  n_iter_          : 22 （各二分类的最大值）
  t_               : 39535 = 39534 + 1

== n_jobs 的作用范围 ==
  '_fit_binary' 源码里出现 n_jobs ? False
  n_jobs=1 与 -1 的 coef_ 完全一致: True
  两者 n_iter_: 10 10


## 8. P6 / P7｜两套停止机制：`tol` 看训练 loss，`early_stopping` 看验证集

**P6（`tol`）**：内循环每个 epoch 维护一个 loss 标量，若 `loss > previous_loss - tol` 就停。它比较的是训练 loss 的下降幅度，不是权重变化；`tol=None` 时阈值变成 `-inf`，等于永不早停，也不会报 `ConvergenceWarning`。

**P7（`early_stopping`）**：用 `StratifiedShuffleSplit` 切出验证集，判据是「验证集 score 连续 `n_iter_no_change` 轮提升不足 `tol`」。它与 P6 是两套独立机制；`partial_fit` 直接禁止早停。

In [10]:
import warnings

import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron

X, y = make_blobs(n_samples=120, centers=2, random_state=0, cluster_std=2.5)

print("== tol 不同取值下的警告 ==")
for tol in (None, 1e-3):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        est = Perceptron(max_iter=3, tol=tol, random_state=0).fit(X, y)
    names = [w.category.__name__ for w in caught]
    print("  tol=%-6r -> n_iter_=%d, 警告=%s" % (tol, est.n_iter_, names or "无"))

print()
early = Perceptron(early_stopping=True, random_state=0, max_iter=1000).fit(X, y)
print("== early_stopping ==")
print("  n_iter_=%d, t_=%d, n_iter_*n+1=%d"
      % (early.n_iter_, int(early.t_), early.n_iter_ * len(y) + 1))

try:
    Perceptron(early_stopping=True).partial_fit(X, y, classes=np.array([0, 1]))
except Exception as e:
    print("  partial_fit + early_stopping ->", type(e).__name__, ":", str(e).splitlines()[0])

== tol 不同取值下的警告 ==
  tol=None   -> n_iter_=3, 警告=无
  tol=0.001  -> n_iter_=3, 警告=['ConvergenceWarning']

== early_stopping ==
  n_iter_=11, t_=1321, n_iter_*n+1=1321
  partial_fit + early_stopping -> ValueError : early_stopping should be False with partial_fit


## 9. P8｜稀疏输入的截距每步只走 1%

`make_dataset` 对稠密输入返回 `intercept_decay=1.0`，对稀疏输入返回 `SPARSE_INTERCEPT_DECAY=0.01`（源码注释：*avoid intercept oscillation*）。

结果：**同一份数据、同一组超参，`csr_matrix` 与稠密数组会得到不同的模型**。感知机默认不加正则，截距被压慢后，系数需要「补偿」它。

In [11]:
import numpy as np
import scipy.sparse as sp
from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron
from sklearn.linear_model._base import SPARSE_INTERCEPT_DECAY, make_dataset

print("SPARSE_INTERCEPT_DECAY =", SPARSE_INTERCEPT_DECAY)

X, y = make_blobs(n_samples=100, centers=2, random_state=3, cluster_std=1.5)
yf = y.astype(np.float64)
sw = np.ones(len(y))
print("make_dataset 的 intercept_decay: dense=%s, sparse=%s"
      % (make_dataset(X, yf, sw)[1], make_dataset(sp.csr_matrix(X), yf, sw)[1]))

kw = dict(shuffle=False, random_state=0, max_iter=30, tol=None)
d = Perceptron(**kw).fit(X, y)
s = Perceptron(**kw).fit(sp.csr_matrix(X), y)

print()
print("%-8s %-24s %-12s %-8s" % ("输入", "coef_ 前两维", "intercept_", "t_"))
for name, est in (("dense", d), ("csr", s)):
    print("%-8s %-24s %-12.2f %-8d"
          % (name, np.round(est.coef_[0][:2], 3), est.intercept_[0], int(est.t_)))
print()
print("coef 是否一致:", np.allclose(d.coef_, s.coef_))

SPARSE_INTERCEPT_DECAY = 0.01
make_dataset 的 intercept_decay: dense=1.0, sparse=0.01

输入       coef_ 前两维                intercept_   t_      
dense    [-9.903 -7.036]          3.00         3001    
csr      [-9.376 -8.868]          0.14         3001    

coef 是否一致: False


## 10. P9 / P10｜权重不是重采样；感知机没有概率

**P9（`class_weight`）**：`class_weight="balanced"` 只算出每类一个**实数**权重，二分类把正、负类权重交给内循环，多分类每个 OvA job 用「本类权重 + 1.0」。数据不复制、行数不变，效果与 `sample_weight` 乘性叠加。

所以它和「复制少数类样本」**不等价**：重采样只能取整数倍，而 `balanced` 给出的是分数权重，两者连权重比例都凑不齐；再加上复制样本还会改变每个 epoch 的样本数与遍历顺序。

**P10（API）**：`Perceptron` 没有 `predict_proba`，只有未校准的 `decision_function`。二分类返回 1D，多分类返回 2D，`predict` 走符号 / `argmax`。

In [12]:
import numpy as np
from sklearn.datasets import load_digits, make_blobs, make_classification
from sklearn.linear_model import Perceptron
from sklearn.utils.class_weight import compute_class_weight

X, y = make_classification(n_samples=300, n_features=2, n_informative=2, n_redundant=0,
                           weights=[0.85, 0.15], class_sep=1.2, flip_y=0.03, random_state=0)
print("== 不平衡数据 ==")
print("  类别分布:", np.bincount(y))
w = compute_class_weight("balanced", classes=np.array([0, 1]), y=y)
print("  balanced 权重:", np.round(w, 4), "-> 比例 1 :", round(w[1] / w[0], 3))

balanced = Perceptron(class_weight="balanced", random_state=0, max_iter=200, tol=None).fit(X, y)
print()
print("  class_weight='balanced'  : coef=%s, train score=%.3f"
      % (np.round(balanced.coef_[0], 3), balanced.score(X, y)))

idx = y == 1
X_dup = np.vstack([X] + [X[idx]] * 3)
y_dup = np.concatenate([y] + [y[idx]] * 3)
dup = Perceptron(random_state=0, max_iter=200, tol=None).fit(X_dup, y_dup)
print("  复制少数类 4 份（比例 1:6）: coef=%s, train score=%.3f"
      % (np.round(dup.coef_[0], 3), dup.score(X, y)))
print()
print("  coef 是否相同:", np.allclose(balanced.coef_, dup.coef_))
print("  -> 重采样只能整数倍，balanced 是分数权重，两者不等价")

try:
    Perceptron(class_weight="balanced").partial_fit(X, y, classes=np.array([0, 1]))
except Exception as e:
    print()
    print("  partial_fit + balanced ->", type(e).__name__, ":", str(e).splitlines()[0][:80])

Xb, yb = make_blobs(n_samples=40, centers=2, random_state=0)
pb = Perceptron(random_state=0).fit(Xb, yb)
print()
print("== API 边界 ==")
print("  有 predict_proba 吗？        :", hasattr(pb, "predict_proba"))
print("  二分类 decision_function.shape:", pb.decision_function(Xb).shape)
print("  二分类 predict.shape          :", pb.predict(Xb).shape)

Xd, yd = load_digits(return_X_y=True)
mdl = Perceptron(random_state=0, max_iter=5, tol=None).fit(Xd, yd)
print("  多分类 decision_function.shape:", mdl.decision_function(Xd).shape)

== 不平衡数据 ==
  类别分布: [252  48]
  balanced 权重: [0.5952 3.125 ] -> 比例 1 : 5.25

  class_weight='balanced'  : coef=[ 8.829 -5.29 ], train score=0.720
  复制少数类 4 份（比例 1:6）: coef=[ 4.621 -1.961], train score=0.843

  coef 是否相同: False
  -> 重采样只能整数倍，balanced 是分数权重，两者不等价

  partial_fit + balanced -> ValueError : class_weight 'balanced' is not supported for partial_fit. In order to use 'balan

== API 边界 ==
  有 predict_proba 吗？        : False
  二分类 decision_function.shape: (40,)
  二分类 predict.shape          : (40,)
  多分类 decision_function.shape: (1797, 10)


## 11. 小结

| # | 现象 | 一句话 |
|---|---|---|
| P1 | 221 行，docstring 占 158 行 | 减法 API：删 `loss`、删 `average`、写死 `constant` |
| P2 | `perceptron → Hinge(0.0)`，`hinge → Hinge(1.0)` | 感知机与线性 SVM 只差一个阈值 |
| P3 | `shuffle=False + tol=None` 时手工循环逐元素一致 | 边界是 `y·f(x) <= 0`，0 分界也会更新 |
| P4 | `t_ = n_iter_ × n_samples + 1` | docstring 说是「更新次数」，实际是「样本数 + 1」 |
| P5 | 多分类 OvA + joblib | `n_jobs` 只在多分类生效，`n_iter_` 取最大值 |
| P6 | `tol` 看训练 loss | `tol=None` 静默跑满 `max_iter`，且不警告 |
| P7 | `early_stopping` 看验证集 | 分层切分在 OvA 之前，所有二分类共享掩码 |
| P8 | 稀疏截距衰减 0.01 | 同一数据 dense 与 csr 得到不同模型（`b` 3.0 vs 0.14） |
| P9 | `class_weight` 是乘性权重 | 不重采样；与复制样本近似等价、非恒等 |
| P10 | 没有 `predict_proba` | 只有未校准的 `decision_function` |

**收尾**：所有「不一样」都能回溯到基类的开关——`loss` / `learning_rate` / `penalty` / `average` / `shuffle` / `tol` / `early_stopping` / 稀疏分支。`Perceptron` 做的事，就是把这些开关固定成课堂上那个感知机；一旦某个开关被打开（例如 `penalty='l1'` 或稀疏输入），它就不再是教科书里的算法了。